In [9]:
%pip install groq
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


instalacao da biblioteca groq que permite o uso da api do llm e dotenv para uso da chave

In [10]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()

True

uso da biblioteca 'os' e 'dotenv' para configurar a chave como variavel de ambiente

In [11]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explique a importância de modelos de linguagem rápidos",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


**Introdução**

Os modelos de linguagem são sistemas de inteligência artificial (IA) projetados para processar e gerar texto humano. Com o avanço da tecnologia e o aumento da demanda por aplicações de IA, os modelos de linguagem rápidos se tornaram cada vez mais importantes. Neste artigo, vamos explorar a importância desses modelos e como eles estão revolucionando a forma como trabalhamos com texto e linguagem.

**O que são modelos de linguagem rápidos?**

Modelos de linguagem rápidos são sistemas de IA projetados para processar e gerar texto em tempo real, ou seja, com alta velocidade e eficiência. Eles utilizam algoritmos e técnicas avançadas de processamento de linguagem natural (PLN) para analisar e gerar texto, permitindo que as aplicações de IA sejam mais rápidas e eficazes.

**Importância dos modelos de linguagem rápidos**

Aqui estão algumas razões pelas quais os modelos de linguagem rápidos são importantes:

1. **Eficiência**: Os modelos de linguagem rápidos permitem que as ap

In [12]:
class Agent:
  def __init__(self, client, system):
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    completion = client.chat.completions.create(
      messages=self.messages,
      model="llama-3.3-70b-versatile"
    )
    return completion.choices[0].message.content



In [13]:
system_prompt_viagem = """
Você opera em um loop de: Pensamento, Ação, PAUSA, Observação.
Ao final do loop, você fornece uma Resposta.

Use 'Pensamento' para descrever suas reflexões sobre a pergunta feita.
Use 'Ação' para executar uma das ferramentas disponíveis - e então retorne PAUSA.
'Observação' será o resultado da execução dessas ações.

Suas ações disponíveis são:

buscar_voo:
Ex: Ação: buscar_voo: Londres
Retorna o preço médio de uma passagem saindo de São Paulo para o destino.

converter_moeda:
Ex: Ação: converter_moeda: 100 USD para BRL
Realiza a conversão de valores entre moedas (usa taxas fixas para o exemplo).

google_search:
Ex: Ação: google_search: Melhores pontos turísticos em Lisboa
Retorna um resumo de busca na web.

Sempre verifique o preço do voo antes de sugerir um orçamento.

Exemplo de sessão:

Pergunta: Quanto custa viajar para Paris em reais?
Pensamento: Preciso primeiro saber o valor da passagem para Paris e depois converter para reais.
Ação: buscar_voo: Paris
PAUSA

Observação: O voo para Paris custa 900 USD.

Pensamento: Agora preciso converter 900 USD para BRL.
Ação: converter_moeda: 900 USD para BRL
PAUSA

Observação: 900 USD equivalem a 4500 BRL.

Resposta: Uma viagem para Paris custa aproximadamente 4500 BRL em passagens.
""".strip()

In [14]:
import re

def buscar_voo(destino):
    # Exemplos de voos
    voos = {
        "roma": "850 USD",
        "toquio": "1200 USD",
        "lisboa": "700 USD",
        "nova york": "600 USD"
    }
    # Retorna o valor do voo. O .lower() garante que a busca não seja case-sensitive.
    # O segundo parâmetro do .get() é a mensagem de erro padrão caso não encontre.
    return voos.get(destino.lower(), "Destino não encontrado no catálogo.")

def converter_moeda(consulta):
    # Regex para encontrar o padrão: "NÚMERO MOEDA_ORIGEM para MOEDA_DESTINO"
    # O (\\d+) captura números, e o (\\w+) captura letras (siglas das moedas).
    match = re.search(r"(\d+) (\w+) para (\w+)", consulta)
    if not match: return "Formato de conversão inválido."
    
    # Extrai os grupos capturados pela Regex
    valor = float(match.group(1))
    moeda_origem = match.group(2).upper()
    moeda_destino = match.group(3).upper()

    # Exemplos de taxas
    taxas = {"USD_BRL": 5.50, "EUR_BRL": 6.0, "BRL_USD": 0.15}
    # Cria a chave de busca (Ex: "USD_BRL")
    chave = f"{moeda_origem}_{moeda_destino}"
    
    # Se a conversão for suportada no dicionário, realiza o cálculo matemático
    if chave in taxas:
        resultado = valor * taxas[chave]
        return f"{valor} {moeda_origem} equivalem a {resultado} {moeda_destino}"
    return "Taxa de câmbio não disponível."

In [15]:
import re

def agent_loop_viagem(max_iterations, query):
    # Instancia o agente com o cliente do Groq e o prompt de sistema
    agent = Agent(client, system_prompt_viagem)
    tools = {
        "buscar_voo": buscar_voo,
        "converter_moeda": converter_moeda
    }
    
    next_prompt = query
    i = 0
    
    # Inicia o loop delimitado pelo número máximo de iterações para evitar loops infinitos
    while i < max_iterations:
        i += 1

        # Chama o LLM passando o prompt atual (Pode ser a query inicial ou uma Observação)
        result = agent(next_prompt)
        print(f"\n--- Iteração {i} ---")
        print(result)

        # Busca por AÇÃO e PAUSA no texto
        if "PAUSA" in result.upper() and "AÇÃO:" in result.upper():
            # Regex para identificar qual ferramenta o LLM escolheu e qual foi o argumento passado.
            # Captura no formato "Ação: nome_da_ferramenta: argumento"
            action_match = re.findall(r"Ação: ([a-z_]+): (.+)", result, re.IGNORECASE)
            
            if action_match:
                # Extrai o nome da ferramenta e o argumento da lista resultante da Regex
                chosen_tool = action_match[0][0].strip()
                arg = action_match[0][1].strip()

                # Verifica se a ferramenta que o LLM pediu realmente existe no nosso dicionário
                if chosen_tool in tools:
                    print(f"Executando ferramenta: {chosen_tool} com argumento: {arg}")
                    # Executa a função Python correspondente passando o argumento
                    result_tool = tools[chosen_tool](arg)
                    # Formata a resposta para enviar de volta ao LLM na próxima iteração
                    next_prompt = f"Observação: {result_tool}"
                else:
                    # Trata o caso de "alucinação" onde o modelo inventa uma ferramenta
                    next_prompt = f"Observação: Erro - A ferramenta {chosen_tool} não existe."
                
                print(next_prompt)
                continue

        if "Resposta:" in result:
            break

In [16]:
agent_loop_viagem(
    max_iterations=5, 
    query="Quero visitar Roma. Quanto vou gastar em reais na passagem?"
)


--- Iteração 1 ---
Pensamento: Para calcular o custo da passagem para Roma em reais, primeiro preciso saber o preço médio de uma passagem saindo de São Paulo para Roma. Depois, terei que converter esse valor para reais, considerando que o preço pode ser em outra moeda.

Ação: buscar_voo: Roma
PAUSA
Executando ferramenta: buscar_voo com argumento: Roma
Observação: 850 USD

--- Iteração 2 ---
Pensamento: Agora que sei que o preço médio da passagem para Roma é de 850 USD, preciso converter esse valor para reais para entender o custo em nossa moeda. Isso exigirá uma conversão de moeda.

Ação: converter_moeda: 850 USD para BRL
PAUSA
Executando ferramenta: converter_moeda com argumento: 850 USD para BRL
Observação: 850.0 USD equivalem a 4675.0 BRL

--- Iteração 3 ---
Pensamento: Com o resultado da conversão, agora tenho o custo da passagem para Roma em reais. Posso usar essa informação para fornecer uma resposta ao usuário sobre o quanto ele gastará em reais na passagem para Roma.

Resposta